# 실전 12-1: Context Engineering (인지 영역 설계)

## 1. Context Engineering이란?
단순히 '말을 예쁘게 하는' 프롬프트 엔지니어링을 넘어, 모델의 뇌(Context Window)에 주입되는 **정보의 양과 질을 시스템적으로 관리**하는 기술입니다.
- 1. **Context Compression (컨텍스트 압축)**: 불필요한 토큰(조사, 중복 단어)을 날려버려서 API 비용을 절감하고, RAG로 검색된 문서 100장을 10장 분량으로 요약해서 던져주는 기술 (예: Microsoft의 LLMLingua)
- 2. **Prompt Caching (프롬프트 캐싱)**: 수만 장의 사내 문서를 매번 LLM에게 읽히면 돈이 수십만 원씩 깨집니다. 이걸 한 번만 읽히고 캐시(Cache)에 저장해 두어, 다음 질문부터는 90% 이상 할인된 가격에 즉시 응답하게 만드는 실무 필수 기술입니다.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

## 2. 실험 1: Context Compression (가상의 압축 체인 만들기)
LLMLingua와 같은 무거운 로컬 모델을 돌리는 대신, 아주 싸고 빠른 소형 모델(예: gpt-4o-mini)을 '압축기(Compressor)'로 앞단에 배치하여 글을 요약한 뒤, 똘똘한 대형 모델(gpt-4o)에게 전달하는 실무 아키텍처를 구현해봅니다.

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 1. 압축기 (싸고 빠른 모델)
compressor_llm = ChatOpenAI(model="gpt-4o-mini")
compress_prompt = ChatPromptTemplate.from_template(
    "다음 긴 텍스트에서 불필요한 조사와 잡담을 모두 제거하고 핵심 키워드와 정보만 20% 분량으로 극단적으로 압축해:\n\n{text}"
)
compressor_chain = compress_prompt | compressor_llm

# 2. 메인 엔진 (비싸고 똑똑한 모델)
main_llm = ChatOpenAI(model="gpt-4o")
main_prompt = ChatPromptTemplate.from_template(
    "다음 압축된 정보를 바탕으로 사용자의 질문에 완벽하게 답해.\n\n[압축된 정보]: {compressed_context}\n\n[질문]: {question}"
)
main_chain = main_prompt | main_llm

# 3. 테스트용 거대한 텍스트
huge_document = """
안녕하십니까. 저는 애플의 CEO 팀 쿡입니다. 
오늘 저희는 여러분께 아주 놀랍고 혁신적인 소식을 전해드리려고 이 자리에 섰습니다. 
우리가 매일 사용하는 스마트폰의 한계를 뛰어넘는 새로운 제품, 바로 아이폰 15 프로를 소개합니다. 
이 제품은 항공우주 등급의 티타늄 디자인을 적용하여 역대 프로 모델 중 가장 가볍고 튼튼합니다. 
또한 A17 Pro 칩을 탑재하여 모바일 게이밍의 새로운 시대를 열었죠. 
카메라 역시 놀랍습니다. 5배 광학 줌을 지원하여 멀리 있는 물체도 눈앞에 있는 것처럼 선명하게 찍을 수 있습니다.
"""

print("=== [1단계: Context 압축] ===")
compressed_result = compressor_chain.invoke({"text": huge_document})
print(compressed_result.content)

print("\n=== [2단계: 메인 엔진 답변] ===")
final_answer = main_chain.invoke({
    "compressed_context": compressed_result.content, 
    "question": "아이폰 15 프로의 주요 스펙 3가지를 요약해줘."
})
print(final_answer.content)

=== [1단계: Context 압축] ===
애플 CEO 팀 쿡, 아이폰 15 프로 소개. 
- 항공우주 등급 티타늄 디자인: 가장 가볍고 튼튼함.
- A17 Pro 칩: 모바일 게이밍 혁신.
- 카메라: 5배 광학 줌 지원.

=== [2단계: 메인 엔진 답변] ===
아이폰 15 프로의 주요 스펙 3가지는 다음과 같습니다:

1. 항공우주 등급 티타늄 디자인으로 가장 가볍고 튼튼한 디자인을 제공합니다.
2. A17 Pro 칩을 탑재하여 모바일 게이밍 혁신을 이루었습니다.
3. 카메라는 5배 광학 줌을 지원합니다.


## 3. 실험 2: Prompt Caching (비용 90% 깎기)
OpenAI와 Anthropic 모두 API에서 '프롬프트 캐싱'을 지원합니다. 
예를 들어 10만 토큰짜리 책 한 권을 챗봇의 System Prompt로 넣어두면 처음 질문할 땐 1달러가 들지만, 다음 질문부터는 캐싱이 적용되어 0.1달러만 들게 됩니다.

(현재 랭체인 환경에서는 Anthropic의 Claude 모델과 OpenAI의 최신 API가 이를 자동으로 지원하며, 이를 활용하기 위해선 System Message의 크기를 키우고 동일한 세션을 유지하는 구조를 짭니다.)

In [3]:
# 주의: 이 셀은 개념 증명용입니다. 실제 과금 로직은 OpenAI API 단에서 자동으로 처리됩니다.
from langchain_core.messages import SystemMessage, HumanMessage

# 매우 거대한 System Message (한 번 읽히면 OpenAI 서버 메모리에 캐싱됨)
huge_system_message = SystemMessage(content="당신은 애플의 사내 규정집 1만 페이지를 숙지한 챗봇입니다. " + ("(규정 내용 블라블라...) " * 1000))

messages_first_turn = [
    huge_system_message,
    HumanMessage(content="휴가 규정 알려줘.")
]
# 첫 번째 호출: System Message를 전부 읽어야 하므로 과금이 크게 발생함
# main_llm.invoke(messages_first_turn)

messages_second_turn = [
    huge_system_message,
    HumanMessage(content="휴가 규정 알려줘."),
    SystemMessage(content="(AI의 이전 답변)"),
    HumanMessage(content="그럼 연차는 이월 돼?")
]
# 두 번째 호출: OpenAI 서버가 앞부분의 huge_system_message가 이전과 똑같다는 걸 인지하고,
# 캐시를 활용하여 새로 추가된 질문 토큰에 대해서만 돈을 받습니다 (비용 90% 감소, 응답 속도 2배 이상 향상)

## 4. 결론
프롬프트 엔지니어링이 "말을 어떻게 할까?" 라면,
컨텍스트 엔지니어링은 **"모델의 뇌 용량(토큰 한도)과 돈(API 비용)을 어떻게 쥐어짜서 최고의 효율을 낼까?"** 를 고민하는 아키텍트의 영역입니다.